# Figure 7 — relative PL maps in $D_{it}$–$Q_f$ space

This notebook reproduces **Figure 7**. The calculated PL of a POx/AlOx-coated surface is normalized to a bare-InP reference with $D_{it}=10^{12}$ eV$^{-1}$ cm$^{-2}$, $Q_f=0$, and no optical coating. White contours show the measured ratios at the optimal 250 °C anneal; the circle marks the normalization point and the star the reported POx/AlOx-on-Si parameters.

This is a comparatively expensive notebook because every map point requires a self-consistent one-dimensional transport solution.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import LogFormatterMathtext
from scipy.constants import h, c
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src'/'surpass').exists()), None)
if root is None:
    raise FileNotFoundError('Could not locate the SURPASS project root.')
sys.path.insert(0, str(root/'src'))

import surpass
from surpass import (
    BulkModel, BulkTransportTable, InterfaceChargeModel, InterfaceDefectModel,
    Layer, OpticalStack, SteadyState1DSolver, SurfaceBoundaryTable,
    SurfaceSRHModel, Wafer, get_optical_material, reconstruct_band_profile,
)

print('surpass version:', surpass.__version__)
print('loaded from:', Path(surpass.__file__).resolve())
if tuple(map(int, surpass.__version__.split('.')[:3])) < (0, 9, 0):
    raise ImportError('This notebook requires surpass 1.0.0 or newer.')

try:
    import scienceplots
    plt.style.use(['science', 'notebook', 'no-latex'])
except ImportError:
    plt.style.use('default')

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10,
    'axes.spines.top': True, 'axes.spines.right': True,
})
output_dir = Path.cwd()/'output'
output_dir.mkdir(parents=True, exist_ok=True)

## 1. Experimental inputs and model switches

Absolute PL from n- and p-InP is not compared because different incident
intensities were used. Every simulated PL map is normalized to a separately
calculated **bare InP** reference of the same wafer polarity, using
$D_{it}=10^{12}$ eV$^{-1}$ cm$^{-2}$ and $Q_f=0$. All points in the map use
the 4-nm PO$_x$/13-nm AlO$_x$ optical stack. The plotted ratio therefore
includes the change in pump coupling caused by adding the dielectric stack,
but remains an internal-PL proxy and does not include stack-dependent
luminescence extraction or detector collection.

The nominal electrostatics omits $Q_{it}$ because the charge-neutrality level
and amphoteric-state distribution of PO$_x$/InP are not independently known.
Setting `INCLUDE_INTERFACE_CHARGE=True` activates an explicitly labelled
sensitivity case; it must not be mistaken for a measured interface model.

The energy-independent capture cross sections
$\sigma_n=10^{-14}$ cm$^2$ and $\sigma_p=10^{-13}$ cm$^2$ originate from the
InP(100) surface-state fit of Adamowicz *et al.*, *Vacuum* **63**, 223--227
(2001), DOI: 10.1016/S0042-207X(01)00195-6. That work considered a chemically
polished InP surface and a U-shaped state distribution. The values are used
here only as literature-motivated starting parameters; they are not measured
capture cross sections for PO$_x$/InP or PO$_x$/AlO$_x$/InP.


In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
wafer_settings = {
    'n-InP': dict(wafer=Wafer('InP', donor_cm3=8.5e18, thickness_um=thickness_um),
                  intensity_w_cm2=1.0, color='tab:blue'),
    'p-InP': dict(wafer=Wafer('InP', acceptor_cm3=5.1e18, thickness_um=thickness_um),
                  intensity_w_cm2=13.0, color='tab:red'),
}

observations = pd.DataFrame([
    {'wafer':'n-InP', 'PL stack/bare':2.82558, 'TRPL bare (ns)':2.55,
     'TRPL stack (ns)':3.40, 'PL after -6e12 corona/before':0.70},
    {'wafer':'p-InP', 'PL stack/bare':0.59693, 'TRPL bare (ns)':1.88,
     'TRPL stack (ns)':1.79, 'PL after -6e12 corona/before':1.30},
])
display(observations)

# Adamowicz et al., Vacuum 63, 223-227 (2001), DOI:
# 10.1016/S0042-207X(01)00195-6. Their fit to a chemically polished
# InP(100) surface used sigma_n=1e-14 and sigma_p=1e-13 cm^2. These are
# literature-motivated starting values, not fitted POx/InP parameters.
DIT_MAP = 3e10
SIGMA_N = 1e-14
SIGMA_P = 1e-13
VTH_N = 4.13e7
VTH_P = 1.51e7
tau_srh = 5e-9 #np.inf
defects = InterfaceDefectModel(
    DIT_MAP, SIGMA_N, SIGMA_P, VTH_N, VTH_P,
    shape='constant', energy_points=201,
)
print(defects.source)

INCLUDE_INTERFACE_CHARGE = True
interface_charge = (InterfaceChargeModel(0.5, 0.02)
                    if INCLUDE_INTERFACE_CHARGE else None)
ENABLE_BGN = True

# Active photon-reabsorption probability used to reduce the net radiative
# carrier loss. Use 0.0 for the conservative no-recycling baseline and 0.93
# for the high-recycling sensitivity case estimated in notebook 08.
# The intrinsic radiative-emission rate used as the PL proxy remains Bnp.
PHOTON_RECYCLING_PROBABILITY = 0.93
if not 0.0 <= PHOTON_RECYCLING_PROBABILITY < 1.0:
    raise ValueError('PHOTON_RECYCLING_PROBABILITY must satisfy 0 <= p_rec < 1')

def surface_model(wafer, charge_cm2, local_defects=defects):
    return SurfaceSRHModel(
        BulkModel(wafer, enable_bgn=ENABLE_BGN), charge_cm2, local_defects,
        electrostatics='self_consistent', driving_force='excess_product',
        interface_charge=interface_charge,
    )

## 4. Optical generation and transport length scales — supplementary candidate

The air/AlO$_x$/PO$_x$/InP transfer-matrix calculation is used for every
passivated map point, with 13 nm AlO$_x$ and 4 nm PO$_x$. A separate air/InP
calculation supplies the genuinely bare optical-generation profile used for
normalization. The film constants remain editable because the bundled
PO$_x$ and AlO$_x$ data are illustrative. This optical calculation determines
pump coupling; it does not yet calculate luminescence extraction.


In [ ]:
air = get_optical_material('air')
inp_optical = get_optical_material('InP_demo')
pox_optical = get_optical_material('POx_demo')
alox_optical = get_optical_material('AlOx_demo')
POX_THICKNESS_NM = 4.0
ALOX_THICKNESS_NM = 13.0
coated_pump = OpticalStack(
    air,
    [Layer(alox_optical, ALOX_THICKNESS_NM, 'AlOx'),
     Layer(pox_optical, POX_THICKNESS_NM, 'POx')],
    inp_optical,
)
bare_pump = OpticalStack(air, [], inp_optical)
optical = coated_pump.solve(wavelength_nm)
bare_optical = bare_pump.solve(wavelength_nm)
depth_um = np.geomspace(1e-4, 100, 500)
uniform_injections = np.logspace(8, 19, 160)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), constrained_layout=True)
for label, settings in wafer_settings.items():
    generation = coated_pump.generation_profile_cm3_s(
        wavelength_nm, settings['intensity_w_cm2'], depth_um*1e-4)
    axes[0].loglog(depth_um, generation, color=settings['color'], label=label)
    bulk = BulkModel(settings['wafer'], enable_bgn=ENABLE_BGN, tau_srh_s=np.inf)
    lengths = [bulk.injection(d).diffusion_length_um for d in uniform_injections]
    axes[1].loglog(uniform_injections, lengths, color=settings['color'], label=label)
axes[0].axvline(optical.absorption_depth_nm/1000, color='black', ls='--', lw=1,
                label=f'1/alpha = {optical.absorption_depth_nm:.0f} nm')
axes[0].set(xlabel='Depth into InP ($\mu$m)', ylabel=r'$G(z)$ (cm$^{-3}$ s$^{-1}$)',
            title='514-nm generation in coated wafer', xlim=(1e-4,2))
axes[0].set_ylim(1e18, 1e26)
axes[1].axhline(optical.absorption_depth_nm/1000, color='black', ls='--', lw=1,
                label='absorption depth')
axes[1].axhline(thickness_um, color='.4', ls=':', lw=1, label='wafer thickness')
axes[1].set(xlabel=r'Uniform $\Delta n$ (cm$^{-3}$)',
            ylabel='Intrinsic diffusion length ($\mu$m)', title='Transport reach')
#for ax in axes:
#    legend = ax.legend(
#        fontsize=10.5,
#        frameon=True,
#        fancybox=True,
#        framealpha=0.85,
#        facecolor='white',
#        edgecolor='0.65',
#        borderpad=0.5,
#    )
#    legend.get_frame().set_linewidth(0.7)
fig.savefig(output_dir/'paper_inp_si_generation_and_lengths.png', bbox_inches='tight')

photon_energy = h*c/(wavelength_nm*1e-9)
display(pd.DataFrame([
    {'structure':'13 nm AlOx / 4 nm POx / InP', 'wafer':label,
     'incident photon flux (cm^-2 s^-1)':s['intensity_w_cm2']/photon_energy,
     'entering photon flux (cm^-2 s^-1)':s['intensity_w_cm2']*optical.substrate_entry_fraction/photon_energy,
     'R':optical.reflectance, 'entry fraction':optical.substrate_entry_fraction,
     'absorption depth (nm)':optical.absorption_depth_nm}
    for label, s in wafer_settings.items()
] + [
    {'structure':'bare InP', 'wafer':label,
     'incident photon flux (cm^-2 s^-1)':s['intensity_w_cm2']/photon_energy,
     'entering photon flux (cm^-2 s^-1)':s['intensity_w_cm2']*bare_optical.substrate_entry_fraction/photon_energy,
     'R':bare_optical.reflectance, 'entry fraction':bare_optical.substrate_entry_fraction,
     'absorption depth (nm)':bare_optical.absorption_depth_nm}
    for label, s in wafer_settings.items()
]))

## 5. Coupled $D_{it}$--$Q_f$ maps at the experimental CW conditions

Every map pixel is a complete one-dimensional steady-state solution for the
4-nm PO$_x$/13-nm AlO$_x$/InP structure. Optical generation, bulk
radiative/Auger recombination, ambipolar diffusion and the nonlinear surface
boundary are solved together. A separate solution for bare air/InP with
$D_{it}=10^{12}$ eV$^{-1}$ cm$^{-2}$ and $Q_f=0$ supplies the normalization
for each wafer polarity. Thus the displayed ratio includes the calculated
change in 514-nm pump coupling on coating, but not a possible change in PL
escape or detector collection.

The selected photon-recycling probability reduces the **net** radiative
carrier loss by the factor $1-p_\mathrm{rec}$, while the internal PL proxy
retains the full intrinsic $Bnp$ emission rate. Consequently, recycling need
not cancel from relative PL ratios when surface and bulk loss fractions change
across the map. At convergence,

$$S_{eff,front}=U_s/\Delta n_s.$$

This operating-point boundary response is not a microscopic velocity and may
become very large when a strong surface sink drives the extrapolated surface
excess density toward zero. Values above $10^8$ cm s$^{-1}$ are therefore
display-clipped as diffusion-limited, while the PL is calculated from the
unclipped flux.


In [ ]:
DIT_POOR = 1e12
DIT_GOOD = 3e10
Q_PATH_END = 2e12
DIT_SILICON = 5e10
QF_SILICON = 4e12
rear_s_cm_s = 1e5
dit_grid = np.unique(np.r_[np.geomspace(1e10,3e12,13),
                           DIT_POOR, DIT_GOOD, DIT_SILICON])
q_grid = np.unique(np.r_[np.linspace(-6e12,6e12,17), QF_SILICON])
edges_cm = np.r_[0.0, np.geomspace(1e-8,5e-4,100),
                 np.linspace(5e-4,thickness_cm,91)[1:]]
coupled_maps = {}
reference_checks = []
failed_points = []

for label, settings in wafer_settings.items():
    bulk = BulkModel(settings['wafer'], enable_bgn=ENABLE_BGN, tau_srh_s=tau_srh)
    transport = BulkTransportTable.from_bulk_model(bulk, 1.0, 1e21, points=161)
    solver = SteadyState1DSolver(transport)

    coated_generation = coated_pump.generation_cell_average_cm3_s(
        wavelength_nm, settings['intensity_w_cm2'], edges_cm)
    bare_generation = bare_pump.generation_cell_average_cm3_s(
        wavelength_nm, settings['intensity_w_cm2'], edges_cm)

    # Bare-air/InP normalization: poor interface, zero fixed charge, and no
    # POx/AlOx in the pump-optics calculation.
    bare_defects = InterfaceDefectModel(
        DIT_POOR, SIGMA_N, SIGMA_P, VTH_N, VTH_P,
        shape='constant', energy_points=151)
    bare_surface = surface_model(settings['wafer'], 0.0, bare_defects)
    bare_boundary = SurfaceBoundaryTable.from_surface_model(
        bare_surface, 1.0, 1e21, points=81)
    bare_result = solver.solve(
        thickness_cm, bare_generation, bare_boundary, rear_s_cm_s,
        cell_edges_cm=edges_cm, max_nfev=1500,
        recycling_probability=PHOTON_RECYCLING_PROBABILITY)
    if not bare_result.converged:
        raise RuntimeError(f'No convergence for bare reference: {label}')
    bare_pl = bare_result.internal_radiative_emission_cm2_s

    arrays = {name:np.empty((dit_grid.size,q_grid.size))
              for name in ('seff','pl','ns','ps','front_delta')}
    for iy, dit in enumerate(dit_grid):
        local_defects = InterfaceDefectModel(
            dit, SIGMA_N, SIGMA_P, VTH_N, VTH_P,
            shape='constant', energy_points=151)
        for ix, charge in enumerate(q_grid):
            smodel = surface_model(settings['wafer'], charge, local_defects)
            boundary = SurfaceBoundaryTable.from_surface_model(
                smodel, 1.0, 1e21, points=81)
            result = solver.solve(
                thickness_cm, coated_generation, boundary, rear_s_cm_s,
                cell_edges_cm=edges_cm, max_nfev=1500,
                recycling_probability=PHOTON_RECYCLING_PROBABILITY)
            if not result.converged:
                failed_points.append((label, dit, charge))
                for name in arrays:
                    arrays[name][iy,ix] = np.nan
                continue
            state = smodel.state(result.front_surface_delta_cm3)
            arrays['seff'][iy,ix] = result.front_effective_s_cm_s
            arrays['pl'][iy,ix] = result.internal_radiative_emission_cm2_s
            arrays['ns'][iy,ix] = state.n_surface_cm3
            arrays['ps'][iy,ix] = state.p_surface_cm3
            arrays['front_delta'][iy,ix] = result.front_surface_delta_cm3

    arrays['bare_reference_pl'] = bare_pl
    arrays['relative_pl'] = arrays['pl']/bare_pl
    iy0 = np.flatnonzero(np.isclose(dit_grid,DIT_POOR,rtol=1e-12))[0]
    ix0 = np.flatnonzero(np.isclose(q_grid,0,atol=1))[0]
    reference_checks.append({
        'wafer':label,
        'p_rec':PHOTON_RECYCLING_PROBABILITY,
        'bare Dit reference':DIT_POOR,
        'bare Q reference':0.0,
        'bare optical layers':'none',
        'coated-map optical layers':'4 nm POx + 13 nm AlOx',
        'bare internal PL':bare_pl,
        'coated PL / bare PL at same electrical coordinates':
            arrays['relative_pl'][iy0,ix0],
    })
    coupled_maps[label] = arrays

display(pd.DataFrame(reference_checks))
if failed_points:
    print('Unconverged extreme map points (left blank in contours):')
    display(pd.DataFrame(failed_points, columns=['wafer','Dit','Qf/q']))


### Compact PL consequence — main-text candidate

White contours mark the measured stack/bare PL ratios. The open circle denotes
the assumed bare-InP electrical reference, $D_{it}=10^{12}$ eV$^{-1}$ cm$^{-2}$
and $Q_f=0$; its normalization is calculated without an optical coating. The
star marks the literature/reference values for PO$_x$/AlO$_x$ on silicon,
$D_{it}=5\times10^{10}$ eV$^{-1}$ cm$^{-2}$ and
$Q_f/q=4\times10^{12}$ cm$^{-2}$. It is shown as a comparison point, not as an
assumption for the InP interface. The central claim is the direction of the
response: chemical passivation alone tends to help both wafers, whereas
positive charge adds a favourable field effect for n-InP and an adverse one
for p-InP.


In [ ]:
measured_ratios = {'n-InP':2.82558, 'p-InP':0.59693}
all_pl = np.concatenate([d['relative_pl'].ravel() for d in coupled_maps.values()])
pl_min, pl_max = max(.2,np.nanmin(all_pl)), np.nanmax(all_pl)
pl_levels = np.linspace(pl_min,pl_max,31)

fig, axes = plt.subplots(1,2,figsize=(11,4.1),sharex=True,sharey=True,
                         constrained_layout=True)
for i,(ax,(label,settings)) in enumerate(zip(axes,wafer_settings.items())):
    rel = coupled_maps[label]['relative_pl']
    cf = ax.contourf(q_grid/1e12,dit_grid,rel,levels=pl_levels,
                     cmap='turbo',extend='both')
    ratio = measured_ratios[label]
    if np.nanmin(rel) <= ratio <= np.nanmax(rel):
        line = ax.contour(q_grid/1e12,dit_grid,rel,levels=[ratio],
                          colors='white',linewidths=1.4)
        ax.clabel(line,fmt={ratio:f'measured {ratio:.3f}x'},
                   colors='white',fontsize=7)
    ax.plot(0,DIT_POOR,'o',mfc='white',mec='black',ms=6,zorder=5,
            label='bare-InP reference' if i == 0 else None)
    ax.plot(QF_SILICON/1e12,DIT_SILICON,'*',mfc='gold',mec='black',mew=.8,
            ms=11,zorder=6,label=r'PO$_x$/AlO$_x$ on Si' if i == 0 else None)
    ax.set_yscale('log')
    ax.set(title=label,xlabel=r'$Q_f/q$ ($10^{12}$ cm$^{-2}$)',
           ylabel=r'$D_{it}$ (eV$^{-1}$ cm$^{-2}$)')

axes[0].legend(loc='best',fontsize=7,frameon=True)
fig.colorbar(cf,ax=axes,label=r'Relative internal PL, '
    r'$\mathrm{PL}_{\mathrm{coated}}(D_{it},Q_f)'
    r'/\mathrm{PL}_{\mathrm{bare}}(D_{it}=10^{12},Q_f=0)$')

fig.savefig(output_dir/'paper_inp_main_coupled_pl_maps.png',bbox_inches='tight')
